# 🎯 SFT Warm-Up for Diagnosis Agent

**Team ClaudeStalkers** — BITS Pilani Hyderabad

**Purpose**: Teach Qwen2.5-7B-Instruct the `COMMAND:/MESSAGE_TO:/MESSAGE:` format *before* running GRPO. Without this warm-up, the base model produces unparseable output → zero reward → GRPO stalls (see `qwen1.5B_output.md`).

**Pipeline**:
1. Build synthetic dataset from heuristic teacher policy (skill=1.0)
2. SFT Qwen2.5-7B-Instruct with LoRA rank 16 for 2 epochs
3. Save adapter to `outputs/war_room_sft/`
4. Feed that adapter into `train_colab.py --sft-checkpoint outputs/war_room_sft`

**Hardware**: A100 recommended (~20 min). T4 works with smaller model (Qwen2.5-1.5B-Instruct) in ~15 min.

In [ ]:
# Cell 1: Clone repo and install deps (~3 min)
!git clone https://github.com/Git4Lokesh/Meta_Hackathon_ClaudeStalkers.git
%cd Meta_Hackathon_ClaudeStalkers
!pip install -q "trl>=0.15.0" "peft>=0.14.0" "transformers>=4.46.0" datasets accelerate bitsandbytes
!pip install -q fastapi pydantic uvicorn matplotlib
!pip install -e . --quiet

In [ ]:
# Cell 2: GPU check
import torch
assert torch.cuda.is_available(), "No GPU! Change Runtime > Change runtime type > GPU"
name = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {name} ({mem:.0f}GB)')

In [ ]:
# Cell 3: Build the SFT dataset from heuristic teacher
import sys
sys.path.insert(0, '.')
import os
os.environ['PYTHONPATH'] = '.'
!PYTHONPATH=. python round2/war_room/build_sft_dataset.py \
    --output outputs/sft_dataset.json \
    --pairs-per-task 50

In [ ]:
# Cell 4: Load dataset and format for SFT
import json
from datasets import Dataset

with open('outputs/sft_dataset.json') as f:
    rows = json.load(f)

print(f'Loaded {len(rows)} training pairs')
print(f'Quality distribution: {dict((q, sum(1 for r in rows if r.get("quality") == q)) for q in ("full", "cmd_only", "msg_only"))}')

# Convert to HuggingFace dataset with chat format
def to_chat(row):
    return {
        'messages': row['prompt'] + row['completion'],
    }

dataset = Dataset.from_list([to_chat(r) for r in rows])
print(f'Dataset ready: {dataset}')
print('\nSample:')
print(dataset[0]['messages'])

In [ ]:
# Cell 5: Load Qwen2.5-7B-Instruct in 4-bit
# NOTE: For T4, change MODEL_NAME to 'Qwen/Qwen2.5-1.5B-Instruct'
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'  # A100: 7B, T4: 1.5B

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
print(f'Model loaded: {MODEL_NAME}')

In [ ]:
# Cell 6: Configure LoRA (rank 16, matches train_colab.py)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 7: Configure and run SFT training (~15-20 min on A100)
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir='outputs/war_room_sft',
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    max_length=2048,
    bf16=True,
    logging_steps=5,
    save_strategy='epoch',
    save_total_limit=1,
    report_to='none',
    seed=42,
)

# TRL >= 0.16 uses processing_class; older uses tokenizer
try:
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=dataset,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=dataset,
        tokenizer=tokenizer,
    )

trainer.train()
trainer.save_model('outputs/war_room_sft')
tokenizer.save_pretrained('outputs/war_room_sft')
print('✅ SFT adapter saved to outputs/war_room_sft/')

In [ ]:
# Cell 8: Verify format compliance on held-out prompts
import re

sys.path.insert(0, '.')
from round2.war_room.environment import WarRoomEnvironment
from round2.war_room.build_sft_dataset import DIAGNOSIS_SYSTEM_PROMPT, TRIAGE_MESSAGES, _build_prompt

def check_format(text):
    has_cmd = 'COMMAND:' in text.upper()
    has_to = 'MESSAGE_TO:' in text.upper()
    has_msg = 'MESSAGE:' in text.upper()
    return has_cmd, has_to, has_msg

# Generate on 10 held-out episodes (seed=1000+, unseen during training)
env = WarRoomEnvironment()
compliant = 0
total = 20

for i in range(total):
    task_id = ['task1', 'task2', 'task3', 'task4'][i % 4]
    obs = env.reset(task_id=task_id, seed=1000 + i)
    prompt_text = _build_prompt(obs.diagnosis.text, task_id)
    messages = [
        {'role': 'system', 'content': DIAGNOSIS_SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt_text},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt',
    ).to(model.device)
    
    with torch.no_grad():
        output = model.generate(
            input_ids, max_new_tokens=200, temperature=0.3, do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    completion = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
    has_cmd, has_to, has_msg = check_format(completion)
    if has_cmd and has_to and has_msg:
        compliant += 1
    if i < 3:
        print(f'--- {task_id} seed={1000+i} ---')
        print(f'Format: CMD={has_cmd} TO={has_to} MSG={has_msg}')
        print(completion[:200])
        print()

pct = 100 * compliant / total
print(f'\n✅ Format compliance: {compliant}/{total} ({pct:.0f}%)')
print(f'{"✅ READY FOR GRPO" if pct >= 60 else "⚠️ Below 60% threshold — consider more SFT epochs"}')

In [ ]:
# Cell 9: Save metrics and download adapter
import json

metrics = {
    'model': MODEL_NAME,
    'dataset_size': len(rows),
    'epochs': 2,
    'format_compliance': pct,
    'loss_history': [h.get('loss', 0) for h in trainer.state.log_history if 'loss' in h],
}
with open('outputs/war_room_sft/sft_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Metrics:')
print(json.dumps(metrics, indent=2))

# Package adapter for download
import shutil
shutil.make_archive('outputs/war_room_sft_adapter', 'zip', 'outputs/war_room_sft')
print('\n✅ Adapter zipped: outputs/war_room_sft_adapter.zip')
print('\nIn Colab, download with:')
print('  from google.colab import files; files.download("outputs/war_room_sft_adapter.zip")')

## Next Step

After SFT completes and format compliance is ≥ 60%, run GRPO on top of this checkpoint:

```bash
!PYTHONPATH=. python round2/war_room/train_colab.py \
    --episodes 30 \
    --tasks task1 task2 task3 \
    --sft-checkpoint outputs/war_room_sft
```

This gives GRPO a policy that already produces valid format, so rewards will be non-zero from step 1.